In [1]:
# ============================================================
# CELL 1: Check Kaggle runtime, GPU, CUDA, Python, inputs
# ============================================================

from pathlib import Path
import os, sys, subprocess, platform, shutil, json, time, glob

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"
KAGGLE_INPUT = Path("/kaggle/input")
SADTALKER_DIR = Path("/kaggle/working/SadTalker")

print("=" * 80)
print("KAGGLE RUNTIME CHECK")
print("=" * 80)
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", Path.cwd())
print("Project directory:", PROJECT)

print("\n" + "=" * 80)
print("GPU / CUDA CHECK")
print("=" * 80)

try:
    import torch
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
        print("GPU count:", torch.cuda.device_count())
    else:
        print("WARNING: GPU not detected. Enable Kaggle Accelerator -> GPU.")
except Exception as e:
    print("Torch import failed:", repr(e))

print("\nNVIDIA-SMI:")
try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception as e:
    print("nvidia-smi unavailable:", repr(e))

print("\n" + "=" * 80)
print("KAGGLE INPUT FILES")
print("=" * 80)

if not KAGGLE_INPUT.exists():
    raise FileNotFoundError("/kaggle/input not found. Are you running this in Kaggle?")

all_input_files = [p for p in KAGGLE_INPUT.rglob("*") if p.is_file()]
if not all_input_files:
    raise FileNotFoundError(
        "No files found in /kaggle/input. Upload a JPG/PNG image, prompt.txt, and optional audio."
    )

for p in sorted(all_input_files):
    try:
        print(" -", p, "|", p.stat().st_size, "bytes")
    except Exception:
        print(" -", p)

print("\nCell 1 complete.")

KAGGLE RUNTIME CHECK
Python executable: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Current working directory: /kaggle/working
Project directory: /kaggle/working/portrait_video_zipless

GPU / CUDA CHECK
Torch version: 2.10.0+cu128
CUDA available: True
Torch CUDA version: 12.8
GPU name: Tesla T4
GPU count: 2

NVIDIA-SMI:
Wed Jun  3 08:41:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|======

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/prompt.txt
/kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/IMG_20260525_102029.jpg
/kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/custom_voice_given_sentence.wav


In [3]:
# ============================================================
# CELL 2: Create folders, locate image and prompt, copy to inputs
# ============================================================

from pathlib import Path
from PIL import Image, ImageOps
import shutil, os, re

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"
KAGGLE_INPUT = Path("/kaggle/input")

for d in [PROJECT, INPUT_DIR, OUTPUT_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print("Ready:", d)

print("\nSearching for uploaded passport/photo image...")

image_exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
image_candidates = []
for ext in image_exts:
    image_candidates.extend(KAGGLE_INPUT.rglob(ext))

if not image_candidates:
    raise FileNotFoundError(
        "No JPG/PNG image found. Upload person.jpg or any passport photo as JPG/PNG."
    )

image_candidates = sorted(
    image_candidates,
    key=lambda p: (
        0 if p.name.lower() in ["person.jpg", "person.jpeg", "person.png"] else 1,
        0 if "person" in p.name.lower() else 1,
        str(p)
    )
)

uploaded_image = image_candidates[0]
person_jpg = INPUT_DIR / "person.jpg"

img = Image.open(uploaded_image)
img = ImageOps.exif_transpose(img).convert("RGB")
img.save(person_jpg, "JPEG", quality=95)

print("Selected source image:", uploaded_image)
print("Copied/converted image to:", person_jpg)
print("Image size:", Image.open(person_jpg).size)
print("Image file size:", person_jpg.stat().st_size, "bytes")

print("\nSearching for prompt.txt...")

prompt_candidates = list(KAGGLE_INPUT.rglob("prompt.txt"))

if not prompt_candidates:
    prompt_candidates = sorted(KAGGLE_INPUT.rglob("*.txt"))

if not prompt_candidates:
    raise FileNotFoundError(
        "prompt.txt not found. Upload prompt.txt to Kaggle input."
    )

uploaded_prompt = sorted(prompt_candidates, key=lambda p: (0 if p.name.lower() == "prompt.txt" else 1, str(p)))[0]
prompt_path = INPUT_DIR / "prompt.txt"
shutil.copy(uploaded_prompt, prompt_path)

prompt_text = prompt_path.read_text(encoding="utf-8", errors="ignore").strip()
if not prompt_text:
    raise ValueError("prompt.txt is empty. Add narration/script text to prompt.txt.")

print("Selected prompt file:", uploaded_prompt)
print("Copied prompt to:", prompt_path)
print("\n" + "=" * 80)
print("PROMPT CONTENT")
print("=" * 80)
print(prompt_text)
print("=" * 80)

print("\nCell 2 complete.")

Ready: /kaggle/working/portrait_video_zipless
Ready: /kaggle/working/portrait_video_zipless/inputs
Ready: /kaggle/working/portrait_video_zipless/outputs
Ready: /kaggle/working/portrait_video_zipless/temp

Searching for uploaded passport/photo image...
Selected source image: /kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/IMG_20260525_102029.jpg
Copied/converted image to: /kaggle/working/portrait_video_zipless/inputs/person.jpg
Image size: (1472, 3264)
Image file size: 1244757 bytes

Searching for prompt.txt...
Selected prompt file: /kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/prompt.txt
Copied prompt to: /kaggle/working/portrait_video_zipless/inputs/prompt.txt

PROMPT CONTENT
A professional presenter smiling naturally, subtle head movement, gentle eye blinking, realistic facial expressions, calm confident speaking style, slight shoulder movement, cinematic soft lighting, natural skin texture, stable identity, smooth camera zoom-in, shallow depth-of-field background

In [4]:
# ============================================================
# CELL 3: Install system and Python dependencies safely
# ============================================================

import sys, subprocess, os, site
from pathlib import Path

def run_cmd(cmd, check=True, env=None):
    print("\n" + "=" * 80)
    print("RUNNING:", " ".join(map(str, cmd)))
    print("=" * 80)
    proc = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env
    )
    print(proc.stdout[-12000:])
    print("Return code:", proc.returncode)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {' '.join(map(str, cmd))}")
    return proc

print("Installing system dependencies...")
run_cmd(["apt-get", "update", "-y", "-qq"], check=False)
run_cmd(["apt-get", "install", "-y", "-qq", "ffmpeg", "git", "wget", "espeak-ng"], check=True)

print("\nUpgrading pip tooling...")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)

print("\nInstalling Python dependencies compatible with Kaggle Python 3.12...")

core_packages = [
    "numpy==1.26.4",
    "scipy",
    "pillow",
    "tqdm",
    "pyyaml",
    "yacs",
    "gdown",
    "safetensors",
    "imageio",
    "imageio-ffmpeg",
    "moviepy==1.0.3",
    "opencv-python-headless",
    "scikit-image",
    "scikit-learn",
    "librosa",
    "soundfile",
    "pydub",
    "kornia",
    "face-alignment",
]

run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + core_packages, check=True)

print("\nInstalling optional enhancement packages.")
print("If these fail, SadTalker will still retry without GFPGAN.")

optional_packages = [
    "facexlib",
    "gfpgan",
    "basicsr",
    "realesrgan",
]

run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + optional_packages, check=False)

print("\nDependency version check:")
for mod in ["numpy", "torch", "torchvision", "cv2", "librosa", "PIL", "imageio", "moviepy"]:
    try:
        if mod == "PIL":
            import PIL
            print("PIL/Pillow:", PIL.__version__)
        elif mod == "cv2":
            import cv2
            print("OpenCV:", cv2.__version__)
        else:
            m = __import__(mod)
            print(mod + ":", getattr(m, "__version__", "imported"))
    except Exception as e:
        print(mod, "import failed:", repr(e))

print("\nCell 3 complete.")

Installing system dependencies...

RUNNING: apt-get update -y -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Return code: 0

RUNNING: apt-get install -y -qq ffmpeg git wget espeak-ng
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 125128 files and directories currently install

In [5]:
# ============================================================
# CELL 4: Clone SadTalker, install safe requirements,
#         download and verify checkpoints correctly
# ============================================================

from pathlib import Path
import os, sys, subprocess, re, shutil, time

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"
SADTALKER_DIR = Path("/kaggle/working/SadTalker")

for d in [PROJECT, INPUT_DIR, OUTPUT_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, check=True, cwd=None, env=None):
    print("\n" + "=" * 90)
    print("RUNNING:", " ".join(map(str, cmd)))
    if cwd:
        print("CWD:", cwd)
    print("=" * 90)

    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    print(proc.stdout[-20000:])
    print("Return code:", proc.returncode)

    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {proc.returncode}: {' '.join(map(str, cmd))}"
        )

    return proc

print("=" * 90)
print("CLONE / VERIFY SADTALKER")
print("=" * 90)

if not SADTALKER_DIR.exists():
    run_cmd([
        "git", "clone", "--depth", "1",
        "https://github.com/OpenTalker/SadTalker.git",
        str(SADTALKER_DIR)
    ], check=True)
else:
    print("SadTalker already exists:", SADTALKER_DIR)

if not (SADTALKER_DIR / "inference.py").exists():
    raise FileNotFoundError(f"SadTalker inference.py not found in {SADTALKER_DIR}")

print("\nSadTalker folder verified:", SADTALKER_DIR)

print("\n" + "=" * 90)
print("INSTALL SADTALKER REQUIREMENTS SAFELY")
print("=" * 90)

requirements = SADTALKER_DIR / "requirements.txt"

if requirements.exists():
    skip_names = {
        "torch",
        "torchvision",
        "torchaudio",
        "numpy",
        "scipy",
        "opencv-python",
        "opencv-contrib-python",
        "opencv-python-headless",
        "pillow",
        "moviepy",
        "basicsr",
        "facexlib",
        "gfpgan",
        "realesrgan",
        "gradio",
        "dlib",
    }

    safe_lines = []

    for line in requirements.read_text(errors="ignore").splitlines():
        raw = line.strip()

        if not raw or raw.startswith("#") or raw.startswith("-"):
            continue

        pkg_name = re.split(r"[<>=!~\[]", raw, maxsplit=1)[0].strip().lower()
        pkg_name = pkg_name.replace("_", "-")

        if pkg_name in skip_names:
            print("Skipping risky/already-handled requirement:", raw)
            continue

        safe_lines.append(raw)

    safe_req = TEMP_DIR / "sadtalker_requirements_safe.txt"
    safe_req.write_text("\n".join(safe_lines) + "\n", encoding="utf-8")

    print("\nSafe requirements file:")
    print(safe_req.read_text())

    if safe_lines:
        run_cmd([
            sys.executable, "-m", "pip", "install",
            "-q", "--no-cache-dir",
            "-r", str(safe_req)
        ], check=False)
else:
    print("No requirements.txt found. Continuing.")

print("\nInstalling/repairing optional enhancer dependencies.")
run_cmd([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir",
    "facexlib", "gfpgan", "basicsr", "realesrgan"
], check=False)

print("\n" + "=" * 90)
print("DOWNLOAD SADTALKER CHECKPOINTS")
print("=" * 90)

ckpt_dir = SADTALKER_DIR / "checkpoints"
gfpgan_weights = SADTALKER_DIR / "gfpgan" / "weights"

ckpt_dir.mkdir(parents=True, exist_ok=True)
gfpgan_weights.mkdir(parents=True, exist_ok=True)

required_main_files = {
    ckpt_dir / "mapping_00109-model.pth.tar": 
        "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar",

    ckpt_dir / "mapping_00229-model.pth.tar": 
        "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar",

    ckpt_dir / "SadTalker_V0.0.2_256.safetensors": 
        "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors",

    ckpt_dir / "SadTalker_V0.0.2_512.safetensors": 
        "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors",
}

optional_gfpgan_files = {
    gfpgan_weights / "alignment_WFLW_4HG.pth":
        "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",

    gfpgan_weights / "detection_Resnet50_Final.pth":
        "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",

    gfpgan_weights / "GFPGANv1.4.pth":
        "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",

    gfpgan_weights / "parsing_parsenet.pth":
        "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
}

download_script = SADTALKER_DIR / "scripts" / "download_models.sh"

print("\nTrying official SadTalker download script first...")

if download_script.exists():
    run_cmd(["bash", str(download_script)], check=False, cwd=SADTALKER_DIR)
else:
    print("Official download script not found:", download_script)

def file_ok(path: Path, min_bytes=100_000):
    return path.exists() and path.is_file() and path.stat().st_size >= min_bytes

def download_with_fallback(path: Path, url: str, min_bytes=100_000):
    path.parent.mkdir(parents=True, exist_ok=True)

    if file_ok(path, min_bytes=min_bytes):
        print("Already valid:", path, "|", path.stat().st_size, "bytes")
        return True

    if path.exists() and path.stat().st_size < min_bytes:
        print("Removing incomplete/small file:", path, "|", path.stat().st_size, "bytes")
        path.unlink()

    print("\nDownloading:")
    print("URL :", url)
    print("DEST:", path)

    proc = run_cmd([
        "wget", "-c",
        "--tries=5",
        "--timeout=60",
        "--no-check-certificate",
        url,
        "-O", str(path)
    ], check=False)

    if file_ok(path, min_bytes=min_bytes):
        print("Download OK:", path, "|", path.stat().st_size, "bytes")
        return True

    print("wget failed or produced incomplete file. Trying curl fallback...")

    if path.exists() and path.stat().st_size < min_bytes:
        path.unlink()

    proc = run_cmd([
        "curl", "-L",
        "--retry", "5",
        "--connect-timeout", "60",
        url,
        "-o", str(path)
    ], check=False)

    if file_ok(path, min_bytes=min_bytes):
        print("Download OK using curl:", path, "|", path.stat().st_size, "bytes")
        return True

    print("FAILED:", path)
    return False

print("\nChecking and repairing main SadTalker checkpoints...")

failed_main = []

for path, url in required_main_files.items():
    ok = download_with_fallback(path, url, min_bytes=100_000)
    if not ok:
        failed_main.append(path)

print("\nChecking and repairing optional GFPGAN weights...")

failed_gfpgan = []

for path, url in optional_gfpgan_files.items():
    ok = download_with_fallback(path, url, min_bytes=100_000)
    if not ok:
        failed_gfpgan.append(path)

print("\n" + "=" * 90)
print("FINAL CHECKPOINT VERIFICATION")
print("=" * 90)

print("\nMain checkpoint files:")
for path in required_main_files:
    print(
        path.relative_to(SADTALKER_DIR),
        "| exists:", path.exists(),
        "| size:", path.stat().st_size if path.exists() else 0,
        "bytes"
    )

print("\nGFPGAN files:")
for path in optional_gfpgan_files:
    print(
        path.relative_to(SADTALKER_DIR),
        "| exists:", path.exists(),
        "| size:", path.stat().st_size if path.exists() else 0,
        "bytes"
    )

checkpoint_files = sorted([p for p in ckpt_dir.rglob("*") if p.is_file()])
gfpgan_files = sorted([p for p in gfpgan_weights.rglob("*") if p.is_file()])

print("\nAll checkpoint files found:", len(checkpoint_files))
for p in checkpoint_files:
    print(" -", p.relative_to(SADTALKER_DIR), "|", p.stat().st_size, "bytes")

print("\nAll GFPGAN weight files found:", len(gfpgan_files))
for p in gfpgan_files:
    print(" -", p.relative_to(SADTALKER_DIR), "|", p.stat().st_size, "bytes")

if failed_main:
    print("\nFAILED MAIN FILES:")
    for p in failed_main:
        print(" -", p)

    raise RuntimeError(
        "Main SadTalker checkpoints are incomplete. "
        "Kaggle internet may be disabled or GitHub release downloads may be blocked. "
        "Enable Internet in Kaggle notebook settings and rerun Cell 4."
    )

if failed_gfpgan:
    print("\nWARNING: Some GFPGAN files are missing.")
    print("This is not fatal because Cell 8 will retry without --enhancer gfpgan.")
    for p in failed_gfpgan:
        print(" -", p)

print("\n" + "=" * 90)
print("SADTALKER CHECKPOINTS READY")
print("=" * 90)

print("SadTalker:", SADTALKER_DIR)
print("Checkpoints:", ckpt_dir)
print("GFPGAN weights:", gfpgan_weights)

print("\nCell 4 complete.")

CLONE / VERIFY SADTALKER

RUNNING: git clone --depth 1 https://github.com/OpenTalker/SadTalker.git /kaggle/working/SadTalker
Cloning into '/kaggle/working/SadTalker'...

Return code: 0

SadTalker folder verified: /kaggle/working/SadTalker

INSTALL SADTALKER REQUIREMENTS SAFELY
Skipping risky/already-handled requirement: numpy==1.23.4
Skipping risky/already-handled requirement: scipy==1.10.1
Skipping risky/already-handled requirement: basicsr==1.4.2
Skipping risky/already-handled requirement: facexlib==0.3.0
Skipping risky/already-handled requirement: gradio
Skipping risky/already-handled requirement: gfpgan

Safe requirements file:
face_alignment==1.3.5
imageio==2.19.3
imageio-ffmpeg==0.4.7
librosa==0.9.2 #
numba
resampy==0.3.1
pydub==0.25.1
kornia==0.6.8
tqdm
yacs==0.1.8
pyyaml
joblib==1.1.0
scikit-image==0.19.3
av
safetensors


RUNNING: /usr/bin/python3 -m pip install -q --no-cache-dir -r /kaggle/working/portrait_video_zipless/temp/sadtalker_requirements_safe.txt
  error: subprocess-

In [6]:
# ============================================================
# CELL 5: Kaggle compatibility patches before inference
# ============================================================

from pathlib import Path
import os, sys, re, site, subprocess

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"
SADTALKER_DIR = Path("/kaggle/working/SadTalker")

if not SADTALKER_DIR.exists():
    raise FileNotFoundError(f"SadTalker folder missing: {SADTALKER_DIR}")

def patch_text_file(path: Path, replacements=None, regex_replacements=None):
    if not path.exists() or not path.is_file():
        return False

    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return False

    old_text = text

    if replacements:
        for old, new in replacements:
            text = text.replace(old, new)

    if regex_replacements:
        for pattern, repl in regex_replacements:
            text = re.sub(pattern, repl, text)

    if text != old_text:
        path.write_text(text, encoding="utf-8")
        print("Patched:", path)
        return True

    return False

print("=" * 80)
print("Creating runtime sitecustomize.py")
print("=" * 80)

sitecustomize_path = Path("/kaggle/working/sitecustomize.py")
sitecustomize_path.write_text("""
# Kaggle compatibility patch for older AI repositories.
try:
    import numpy as np

    if not hasattr(np, "VisibleDeprecationWarning"):
        try:
            np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
        except Exception:
            np.VisibleDeprecationWarning = Warning

    if not hasattr(np, "float"):
        np.float = float

    if not hasattr(np, "int"):
        np.int = int

    if not hasattr(np, "bool"):
        np.bool = bool

    if not hasattr(np, "complex"):
        np.complex = complex

except Exception:
    pass

try:
    import torch
    if not getattr(torch.load, "_kaggle_compat_patched", False):
        _original_torch_load = torch.load

        def _torch_load_compat(*args, **kwargs):
            if "weights_only" not in kwargs:
                kwargs["weights_only"] = False
            return _original_torch_load(*args, **kwargs)

        _torch_load_compat._kaggle_compat_patched = True
        torch.load = _torch_load_compat
except Exception:
    pass
""", encoding="utf-8")

print("Created:", sitecustomize_path)
print(sitecustomize_path.read_text())

os.environ["PYTHONPATH"] = "/kaggle/working:" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH:", os.environ["PYTHONPATH"])

print("\n" + "=" * 80)
print("Patching SadTalker source files for NumPy compatibility")
print("=" * 80)

regex_replacements = [
    (r'(?<![\w.])np\.float(?![\w])', 'float'),
    (r'(?<![\w.])np\.int(?![\w])', 'int'),
    (r'(?<![\w.])np\.bool(?![\w])', 'bool'),
    (r'(?<![\w.])np\.complex(?![\w])', 'complex'),
    (r'np\.VisibleDeprecationWarning', 'getattr(np, "VisibleDeprecationWarning", Warning)'),
]

patched_count = 0
for py_file in SADTALKER_DIR.rglob("*.py"):
    if patch_text_file(py_file, regex_replacements=regex_replacements):
        patched_count += 1

print("SadTalker Python files patched:", patched_count)

specific_awing = SADTALKER_DIR / "src" / "face3d" / "util" / "my_awing_arch.py"
if specific_awing.exists():
    patch_text_file(specific_awing, regex_replacements=regex_replacements)
    print("Verified specific np.float patch target:", specific_awing)
else:
    print("WARNING: Specific my_awing_arch.py not found:", specific_awing)

print("\n" + "=" * 80)
print("Patching BasicSR / Real-ESRGAN torchvision import")
print("=" * 80)

basicsr_patterns = []
for root in site.getsitepackages() + [site.getusersitepackages(), "/usr/local/lib"]:
    root_path = Path(root)
    if root_path.exists():
        basicsr_patterns.extend(root_path.rglob("basicsr/data/degradations.py"))

basicsr_patterns = sorted(set(basicsr_patterns))

if not basicsr_patterns:
    print("No BasicSR degradations.py found yet. This is OK if GFPGAN is unavailable.")
else:
    for f in basicsr_patterns:
        patch_text_file(
            f,
            replacements=[
                (
                    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
                    "from torchvision.transforms.functional import rgb_to_grayscale"
                )
            ]
        )
        print("Verified BasicSR file:", f)

print("\n" + "=" * 80)
print("Runtime import verification")
print("=" * 80)

try:
    import numpy as np
    print("NumPy:", np.__version__)
    print("Has np.VisibleDeprecationWarning:", hasattr(np, "VisibleDeprecationWarning"))
    print("np.float available after runtime patch:", hasattr(np, "float"))
    print("np.int available after runtime patch:", hasattr(np, "int"))
    print("np.bool available after runtime patch:", hasattr(np, "bool"))
    print("np.complex available after runtime patch:", hasattr(np, "complex"))
except Exception as e:
    print("NumPy check failed:", repr(e))

try:
    import torch, torchvision
    print("Torch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("Torch/Torchvision check failed:", repr(e))

print("\nCell 5 complete.")

Creating runtime sitecustomize.py
Created: /kaggle/working/sitecustomize.py

# Kaggle compatibility patch for older AI repositories.
try:
    import numpy as np

    if not hasattr(np, "VisibleDeprecationWarning"):
        try:
            np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
        except Exception:
            np.VisibleDeprecationWarning = Warning

    if not hasattr(np, "float"):
        np.float = float

    if not hasattr(np, "int"):
        np.int = int

    if not hasattr(np, "bool"):
        np.bool = bool

    if not hasattr(np, "complex"):
        np.complex = complex

except Exception:
    pass

try:
    import torch
    if not getattr(torch.load, "_kaggle_compat_patched", False):
        _original_torch_load = torch.load

        def _torch_load_compat(*args, **kwargs):
            if "weights_only" not in kwargs:
                kwargs["weights_only"] = False
            return _original_torch_load(*args, **kwargs)

        _torch_load_c

In [7]:
# ============================================================
# CELL 6: Prepare narration audio
# ============================================================

from pathlib import Path
import subprocess, os, re, sys, json, shutil

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"
KAGGLE_INPUT = Path("/kaggle/input")

prompt_path = INPUT_DIR / "prompt.txt"
if not prompt_path.exists():
    raise FileNotFoundError(f"Missing prompt file: {prompt_path}")

prompt_text = prompt_path.read_text(encoding="utf-8", errors="ignore").strip()
if not prompt_text:
    raise ValueError("prompt.txt is empty.")

def run_cmd(cmd, check=True):
    print("\n" + "=" * 80)
    print("RUNNING:", " ".join(map(str, cmd)))
    print("=" * 80)
    proc = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print(proc.stdout[-12000:])
    print("Return code:", proc.returncode)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {' '.join(map(str, cmd))}")
    return proc

audio_exts = ["*.wav", "*.mp3", "*.m4a", "*.flac", "*.aac", "*.ogg", "*.WAV", "*.MP3", "*.M4A", "*.FLAC"]
audio_candidates = []
for ext in audio_exts:
    audio_candidates.extend(KAGGLE_INPUT.rglob(ext))

audio_candidates = sorted(
    audio_candidates,
    key=lambda p: (
        0 if p.name.lower() in ["audio.wav", "audio.mp3", "audio.m4a", "audio.flac"] else 1,
        str(p)
    )
)

narration_wav = OUTPUT_DIR / "narration.wav"
tmp_audio = TEMP_DIR / "fallback_espeak_raw.wav"
speech_text_file = TEMP_DIR / "fallback_narration_text.txt"

if narration_wav.exists():
    narration_wav.unlink()

if audio_candidates:
    uploaded_audio = audio_candidates[0]
    print("Uploaded audio found:", uploaded_audio)

    run_cmd([
        "ffmpeg", "-y",
        "-i", str(uploaded_audio),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-sample_fmt", "s16",
        str(narration_wav)
    ], check=True)

else:
    print("No uploaded audio found. Generating fallback narration using espeak-ng.")

    narration_text = re.split(r"negative\\s*prompt\\s*:", prompt_text, flags=re.I)[0].strip()
    narration_text = re.sub(r"\\s+", " ", narration_text).strip()

    if len(narration_text) < 5:
        narration_text = (
            "Hello. I am presenting this content in a clear and professional manner, "
            "with natural expressions and smooth speech."
        )

    if len(narration_text) > 3000:
        narration_text = narration_text[:3000]

    speech_text_file.write_text(narration_text, encoding="utf-8")

    print("\nFallback narration text:")
    print(narration_text)

    run_cmd([
        "espeak-ng",
        "-v", "en",
        "-s", "145",
        "-p", "45",
        "-f", str(speech_text_file),
        "-w", str(tmp_audio)
    ], check=True)

    run_cmd([
        "ffmpeg", "-y",
        "-i", str(tmp_audio),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-sample_fmt", "s16",
        str(narration_wav)
    ], check=True)

if not narration_wav.exists() or narration_wav.stat().st_size == 0:
    raise RuntimeError(f"Failed to create narration audio: {narration_wav}")

duration_proc = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(narration_wav)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

duration_text = duration_proc.stdout.strip()
print("\n" + "=" * 80)
print("NARRATION AUDIO READY")
print("=" * 80)
print("Path:", narration_wav)
print("Size:", narration_wav.stat().st_size, "bytes")
print("Duration seconds:", duration_text if duration_text else "unknown")

print("\nCell 6 complete.")

Uploaded audio found: /kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/custom_voice_given_sentence.wav

RUNNING: ffmpeg -y -i /kaggle/input/datasets/shalini286r/new-ai-voice-dwarakan/custom_voice_given_sentence.wav -vn -ac 1 -ar 16000 -sample_fmt s16 /kaggle/working/portrait_video_zipless/outputs/narration.wav
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --en

In [8]:
# ============================================================
# CELL 7: Clean and standardize source image
# ============================================================

from pathlib import Path
from PIL import Image, ImageOps
import os

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"

person_jpg = INPUT_DIR / "person.jpg"
person_clean = INPUT_DIR / "person_clean.jpg"

if not person_jpg.exists():
    raise FileNotFoundError(f"Missing source image: {person_jpg}")

img = Image.open(person_jpg)
img = ImageOps.exif_transpose(img).convert("RGB")

original_size = img.size
max_side = 1024

if max(img.size) > max_side:
    img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)

img.save(person_clean, "JPEG", quality=95, optimize=True)

if not person_clean.exists() or person_clean.stat().st_size == 0:
    raise RuntimeError(f"Failed to create clean image: {person_clean}")

print("=" * 80)
print("IMAGE READY")
print("=" * 80)
print("Original image:", person_jpg)
print("Original size:", original_size)
print("Clean image:", person_clean)
print("Clean size:", Image.open(person_clean).size)
print("Clean file size:", person_clean.stat().st_size, "bytes")
print("Note: Image was resized only by aspect-ratio-preserving scaling. No face distortion was applied.")

print("\nCell 7 complete.")

IMAGE READY
Original image: /kaggle/working/portrait_video_zipless/inputs/person.jpg
Original size: (1472, 3264)
Clean image: /kaggle/working/portrait_video_zipless/inputs/person_clean.jpg
Clean size: (462, 1024)
Clean file size: 148940 bytes
Note: Image was resized only by aspect-ratio-preserving scaling. No face distortion was applied.

Cell 7 complete.


In [9]:
from pathlib import Path

log_path = Path("/kaggle/working/portrait_video_zipless/outputs/sadtalker_error_log.txt")
log_path.parent.mkdir(parents=True, exist_ok=True)

if not log_path.exists():
    log_path.write_text(
        "No SadTalker error log was created. Continue with Wav2Lip fallback.\n",
        encoding="utf-8"
    )

print("Dummy log created:", log_path)

Dummy log created: /kaggle/working/portrait_video_zipless/outputs/sadtalker_error_log.txt


In [10]:
# ============================================================
# DIAGNOSTIC CELL: Extract the REAL SadTalker error
# ============================================================

from pathlib import Path
import re

log_path = Path("/kaggle/working/portrait_video_zipless/outputs/sadtalker_error_log.txt")

if not log_path.exists():
    raise FileNotFoundError(f"SadTalker log not found: {log_path}")

text = log_path.read_text(errors="ignore")
lines = text.splitlines()

print("=" * 100)
print("SADTALKER LOG SUMMARY")
print("=" * 100)
print("Log path:", log_path)
print("Total lines:", len(lines))

print("\n" + "=" * 100)
print("LAST 250 LINES")
print("=" * 100)
print("\n".join(lines[-250:]))

print("\n" + "=" * 100)
print("TRACEBACK BLOCKS FOUND")
print("=" * 100)

traceback_indices = [i for i, line in enumerate(lines) if "Traceback (most recent call last)" in line]

if traceback_indices:
    for n, idx in enumerate(traceback_indices[-5:], start=1):
        print("\n" + "-" * 100)
        print(f"TRACEBACK #{n}")
        print("-" * 100)
        print("\n".join(lines[idx:idx + 80]))
else:
    print("No Python traceback block found.")

print("\n" + "=" * 100)
print("AUTOMATIC ERROR CLASSIFICATION")
print("=" * 100)

lower = text.lower()

checks = [
    ("unrecognized arguments", "Unsupported SadTalker command-line argument."),
    ("no face", "Face detection failed."),
    ("face detect", "Face detection failed."),
    ("landmark", "Face landmark detection failed."),
    ("cuda out of memory", "CUDA memory error."),
    ("outofmemoryerror", "CUDA memory error."),
    ("np.float", "NumPy alias compatibility error."),
    ("np.int", "NumPy alias compatibility error."),
    ("visibledeprecationwarning", "NumPy VisibleDeprecationWarning compatibility error."),
    ("functional_tensor", "BasicSR / torchvision functional_tensor import error."),
    ("weights_only", "PyTorch 2.6+ torch.load weights_only error."),
    ("filenotfounderror", "Missing file or checkpoint."),
    ("no such file or directory", "Missing file or checkpoint."),
    ("safetensors", "Safetensors/checkpoint loading issue."),
    ("gfpgan", "GFPGAN enhancement issue."),
    ("basicsr", "BasicSR enhancement issue."),
    ("cannot import name", "Dependency import/version issue."),
    ("module not found", "Missing Python package."),
    ("modulenotfounderror", "Missing Python package."),
]

found = False
for key, explanation in checks:
    if key in lower:
        print("Detected:", explanation)
        found = True

if not found:
    print("No common error pattern detected. Check the TRACEBACK block above.")

print("\nIMPORTANT:")
print("Copy the last traceback block above if you need further debugging.")

SADTALKER LOG SUMMARY
Log path: /kaggle/working/portrait_video_zipless/outputs/sadtalker_error_log.txt
Total lines: 1

LAST 250 LINES
No SadTalker error log was created. Continue with Wav2Lip fallback.

TRACEBACK BLOCKS FOUND
No Python traceback block found.

AUTOMATIC ERROR CLASSIFICATION
No common error pattern detected. Check the TRACEBACK block above.

IMPORTANT:
Copy the last traceback block above if you need further debugging.


In [11]:
# ============================================================
# DYNAMIC FALLBACK CELL: Wav2Lip talking video from single photo
# Produces non-static lip-synced output_raw.mp4
# ============================================================

from pathlib import Path
import os, sys, subprocess, shutil, re, site
from PIL import Image, ImageOps
import cv2
import numpy as np

PROJECT = Path("/kaggle/working/portrait_video_zipless")
INPUT_DIR = PROJECT / "inputs"
OUTPUT_DIR = PROJECT / "outputs"
TEMP_DIR = PROJECT / "temp"

source_image = INPUT_DIR / "person_clean.jpg"
audio_wav = OUTPUT_DIR / "narration.wav"

WAV2LIP_DIR = Path("/kaggle/working/Wav2Lip")
raw_video = OUTPUT_DIR / "output_raw.mp4"
wav2lip_input_video = TEMP_DIR / "wav2lip_static_input.mp4"
safe_face_image = TEMP_DIR / "wav2lip_face_safe.jpg"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("DYNAMIC Wav2Lip FALLBACK: PHOTO + AUDIO → TALKING MP4")
print("=" * 100)

for p in [source_image, audio_wav]:
    print(p, "| exists:", p.exists(), "| size:", p.stat().st_size if p.exists() else 0)
    if not p.exists():
        raise FileNotFoundError(p)

# ------------------------------------------------------------
# Remove old static/fallback outputs
# ------------------------------------------------------------

for p in [
    OUTPUT_DIR / "output.mp4",
    OUTPUT_DIR / "output_raw.mp4",
    wav2lip_input_video,
    safe_face_image,
]:
    if p.exists():
        p.unlink()
        print("Deleted old file:", p)

# ------------------------------------------------------------
# Install dependencies
# ------------------------------------------------------------

def run(cmd, check=True, cwd=None):
    print("\n" + "=" * 90)
    print("RUNNING:", " ".join(map(str, cmd)))
    if cwd:
        print("CWD:", cwd)
    print("=" * 90)

    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(proc.stdout[-15000:])
    print("Return code:", proc.returncode)

    if check and proc.returncode != 0:
        raise RuntimeError("Command failed: " + " ".join(map(str, cmd)))

    return proc

run(["apt-get", "update", "-y", "-qq"], check=False)
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "git", "wget"], check=True)

run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "pip", "setuptools", "wheel"
], check=True)

run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "numpy==1.26.4",
    "opencv-python-headless",
    "librosa",
    "numba",
    "scipy",
    "tqdm",
    "ffmpeg-python",
    "huggingface_hub",
    "hf_xet",
    "gdown"
], check=True)

# ------------------------------------------------------------
# Clone Wav2Lip code
# ------------------------------------------------------------

if not WAV2LIP_DIR.exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/Rudrabha/Wav2Lip.git",
        str(WAV2LIP_DIR)
    ], check=True)
else:
    print("Wav2Lip already exists:", WAV2LIP_DIR)

if not (WAV2LIP_DIR / "inference.py").exists():
    raise FileNotFoundError("Wav2Lip inference.py not found.")

# ------------------------------------------------------------
# Runtime compatibility patches
# ------------------------------------------------------------

sitecustomize_path = Path("/kaggle/working/sitecustomize.py")

sitecustomize_path.write_text("""
try:
    import numpy as np

    if not hasattr(np, "float"):
        np.float = float
    if not hasattr(np, "int"):
        np.int = int
    if not hasattr(np, "bool"):
        np.bool = bool
    if not hasattr(np, "complex"):
        np.complex = complex
    if not hasattr(np, "VisibleDeprecationWarning"):
        try:
            np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
        except Exception:
            np.VisibleDeprecationWarning = Warning
except Exception:
    pass

try:
    import torch

    if not getattr(torch.load, "_kaggle_compat_patched", False):
        _original_torch_load = torch.load

        def _torch_load_compat(*args, **kwargs):
            if "weights_only" not in kwargs:
                kwargs["weights_only"] = False
            return _original_torch_load(*args, **kwargs)

        _torch_load_compat._kaggle_compat_patched = True
        torch.load = _torch_load_compat
except Exception:
    pass
""", encoding="utf-8")

os.environ["PYTHONPATH"] = "/kaggle/working:" + os.environ.get("PYTHONPATH", "")

def patch_file(path, replacements=None, regex_replacements=None):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return False

    text = path.read_text(encoding="utf-8", errors="ignore")
    old = text

    if replacements:
        for a, b in replacements:
            text = text.replace(a, b)

    if regex_replacements:
        for a, b in regex_replacements:
            text = re.sub(a, b, text)

    if text != old:
        path.write_text(text, encoding="utf-8")
        print("Patched:", path)
        return True

    return False

regex_replacements = [
    (r'(?<![\w.])np\.float(?![\w])', 'float'),
    (r'(?<![\w.])np\.int(?![\w])', 'int'),
    (r'(?<![\w.])np\.bool(?![\w])', 'bool'),
    (r'(?<![\w.])np\.complex(?![\w])', 'complex'),
]

for py_file in WAV2LIP_DIR.rglob("*.py"):
    patch_file(py_file, regex_replacements=regex_replacements)

# Fix older librosa usage if present
for py_file in WAV2LIP_DIR.rglob("*.py"):
    patch_file(
        py_file,
        replacements=[
            ("librosa.filters.mel(hp.sample_rate, hp.n_fft", "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft"),
        ]
    )

# ------------------------------------------------------------
# Download Wav2Lip checkpoint
# ------------------------------------------------------------

ckpt_dir = WAV2LIP_DIR / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

wav2lip_ckpt = ckpt_dir / "wav2lip_gan.pth"

if not wav2lip_ckpt.exists() or wav2lip_ckpt.stat().st_size < 100_000_000:
    print("\nDownloading wav2lip_gan.pth from Hugging Face mirror...")

    download_code = f"""
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

target = Path(r"{wav2lip_ckpt}")
target.parent.mkdir(parents=True, exist_ok=True)

p = hf_hub_download(
    repo_id="rippertnt/wav2lip",
    filename="checkpoints/wav2lip_gan.pth",
    local_dir=r"{WAV2LIP_DIR}",
    local_dir_use_symlinks=False
)

print("Downloaded:", p)
print("Target:", target)
"""

    proc = subprocess.run(
        [sys.executable, "-c", download_code],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(proc.stdout[-12000:])

if not wav2lip_ckpt.exists() or wav2lip_ckpt.stat().st_size < 100_000_000:
    print("Hugging Face download failed. Trying Google Drive fallback...")

    run([
        sys.executable, "-m", "gdown",
        "--id", "1dwHujX7RVNCvdzXwhJmiZEoSR2z2bpEx",
        "-O", str(wav2lip_ckpt)
    ], check=False)

if not wav2lip_ckpt.exists() or wav2lip_ckpt.stat().st_size < 100_000_000:
    raise RuntimeError(
        "wav2lip_gan.pth could not be downloaded. "
        "Enable Kaggle Internet and rerun this cell."
    )

print("Wav2Lip checkpoint ready:", wav2lip_ckpt, wav2lip_ckpt.stat().st_size, "bytes")

# ------------------------------------------------------------
# Download S3FD face detector checkpoint
# ------------------------------------------------------------

s3fd_path = WAV2LIP_DIR / "face_detection" / "detection" / "sfd" / "s3fd.pth"
s3fd_path.parent.mkdir(parents=True, exist_ok=True)

if not s3fd_path.exists() or s3fd_path.stat().st_size < 10_000_000:
    run([
        "wget", "-c",
        "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth",
        "-O", str(s3fd_path)
    ], check=True)

print("S3FD detector ready:", s3fd_path, s3fd_path.stat().st_size, "bytes")

# ------------------------------------------------------------
# Create face-safe portrait image
# ------------------------------------------------------------

img = Image.open(source_image)
img = ImageOps.exif_transpose(img).convert("RGB")

w, h = img.size

# White square canvas keeps passport photo centered and detectable
canvas_size = max(720, w, h)
canvas = Image.new("RGB", (canvas_size, canvas_size), (255, 255, 255))
canvas.paste(img, ((canvas_size - w) // 2, (canvas_size - h) // 2))

canvas = canvas.resize((720, 720), Image.Resampling.LANCZOS)
canvas.save(safe_face_image, "JPEG", quality=95)

print("Safe face image:", safe_face_image)
print("Safe image size:", Image.open(safe_face_image).size)

# ------------------------------------------------------------
# Get audio duration
# ------------------------------------------------------------

duration_proc = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_wav)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

audio_duration = float(duration_proc.stdout.strip())
print("Audio duration:", audio_duration)

# ------------------------------------------------------------
# Make temporary static video input for Wav2Lip
# Wav2Lip will animate mouth region from audio.
# ------------------------------------------------------------

run([
    "ffmpeg", "-y",
    "-loop", "1",
    "-i", str(safe_face_image),
    "-i", str(audio_wav),
    "-t", str(audio_duration),
    "-vf", "fps=25,format=yuv420p",
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "18",
    "-c:a", "aac",
    "-b:a", "192k",
    "-shortest",
    str(wav2lip_input_video)
], check=True)

if not wav2lip_input_video.exists() or wav2lip_input_video.stat().st_size == 0:
    raise RuntimeError("Failed to create Wav2Lip input video.")

print("Wav2Lip input video:", wav2lip_input_video)

# ------------------------------------------------------------
# Run Wav2Lip inference
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working:" + env.get("PYTHONPATH", "")

wav2lip_out = OUTPUT_DIR / "wav2lip_dynamic_raw.mp4"

cmd = [
    sys.executable,
    "inference.py",
    "--checkpoint_path", str(wav2lip_ckpt),
    "--face", str(wav2lip_input_video),
    "--audio", str(audio_wav),
    "--outfile", str(wav2lip_out),
    "--pads", "0", "20", "0", "0",
    "--resize_factor", "1",
    "--nosmooth"
]

print("\n" + "=" * 100)
print("RUNNING WAV2LIP")
print("=" * 100)
print(" ".join(cmd))

proc = subprocess.run(
    cmd,
    cwd=str(WAV2LIP_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env
)

print(proc.stdout[-30000:])
print("Return code:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Wav2Lip inference failed. Check the printed log above.")

if not wav2lip_out.exists() or wav2lip_out.stat().st_size == 0:
    raise RuntimeError("Wav2Lip did not create output MP4.")

# ------------------------------------------------------------
# Verify motion: reject truly static output
# ------------------------------------------------------------

def motion_score(video_path, samples=30):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return 0.0, 0.0, 0, 0.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    duration = total_frames / fps if fps else 0

    indices = np.linspace(0, max(total_frames - 1, 0), min(samples, max(total_frames, 1)), dtype=int)

    prev = None
    diffs = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (160, 160))

        if prev is not None:
            diffs.append(float(np.mean(cv2.absdiff(prev, gray))))

        prev = gray

    cap.release()

    if not diffs:
        return 0.0, 0.0, total_frames, duration

    return float(np.mean(diffs)), float(np.max(diffs)), total_frames, duration

mean_motion, max_motion, frames, duration = motion_score(wav2lip_out)

print("\n" + "=" * 100)
print("WAV2LIP MOTION CHECK")
print("=" * 100)
print("Output:", wav2lip_out)
print("Size:", wav2lip_out.stat().st_size, "bytes")
print("Frames:", frames)
print("Duration:", duration)
print("Mean motion:", mean_motion)
print("Max motion:", max_motion)

if frames < 10 or duration <= 1:
    raise RuntimeError("Wav2Lip output is too short or invalid.")

# Wav2Lip motion can be localized around mouth, so threshold is mild.
if max_motion <= 0.15:
    raise RuntimeError(
        "Wav2Lip output appears static. Try a clearer front-facing image with visible mouth/chin."
    )

shutil.copy(wav2lip_out, raw_video)

print("\n" + "=" * 100)
print("DYNAMIC OUTPUT_RAW READY")
print("=" * 100)
print("Copied to:", raw_video)
print("Size:", raw_video.stat().st_size, "bytes")

print("\nNow run Cell 9 and Cell 10.")

DYNAMIC Wav2Lip FALLBACK: PHOTO + AUDIO → TALKING MP4
/kaggle/working/portrait_video_zipless/inputs/person_clean.jpg | exists: True | size: 148940
/kaggle/working/portrait_video_zipless/outputs/narration.wav | exists: True | size: 221284

RUNNING: apt-get update -y -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Return code: 0

RUNNING: apt-get install -y -qq ffmpeg git wget

Return code: 0

RUNNING: /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel
/kaggle/working/sitecustomize.py:18: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "bool"):

Return code: 0

RUNNING: /usr/bin/python3 -m pip install -q --no-cache-dir numpy==1.26.4 opencv-python-headless librosa numba scipy tqdm ffmpeg-python huggingface_hub hf_xet gdown
/kaggle/working/sitecustomize.py:18: FutureWar

In [12]:
# ============================================================
# CELL 9: Render final 1920x1080 HD MP4 with blurred background
# ============================================================

from pathlib import Path
import subprocess, os, sys

PROJECT = Path("/kaggle/working/portrait_video_zipless")
OUTPUT_DIR = PROJECT / "outputs"

raw_video = OUTPUT_DIR / "output_raw.mp4"
final_video = OUTPUT_DIR / "output.mp4"
log_path = OUTPUT_DIR / "sadtalker_error_log.txt"

if not raw_video.exists() or raw_video.stat().st_size == 0:
    print("ERROR: output_raw.mp4 missing or empty:", raw_video)

    if log_path.exists():
        log_text = log_path.read_text(errors="ignore")
        print("\nSadTalker log tail:")
        print("\n".join(log_text.splitlines()[-250:]))

    raise FileNotFoundError(
        "Missing output_raw.mp4. Cell 8 must successfully generate SadTalker animation first."
    )

if final_video.exists():
    final_video.unlink()

filter_complex = (
    "[0:v]"
    "scale=1920:1080:force_original_aspect_ratio=increase,"
    "crop=1920:1080,"
    "gblur=sigma=24,"
    "setsar=1[bg];"
    "[0:v]"
    "scale=1080:1080:force_original_aspect_ratio=decrease,"
    "setsar=1[fg];"
    "[bg][fg]"
    "overlay=(W-w)/2:(H-h)/2,"
    "fps=30,"
    "format=yuv420p[v]"
)

cmd = [
    "ffmpeg", "-y",
    "-i", str(raw_video),
    "-filter_complex", filter_complex,
    "-map", "[v]",
    "-map", "0:a?",
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "18",
    "-c:a", "aac",
    "-b:a", "192k",
    "-ar", "44100",
    "-shortest",
    "-movflags", "+faststart",
    str(final_video)
]

print("=" * 80)
print("RENDERING FINAL HD VIDEO")
print("=" * 80)
print("Command:", " ".join(cmd))

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(proc.stdout[-15000:])
print("Return code:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("FFmpeg final rendering failed.")

if not final_video.exists() or final_video.stat().st_size == 0:
    raise RuntimeError(f"Final output.mp4 missing or empty: {final_video}")

duration_proc = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(final_video)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n" + "=" * 80)
print("FINAL VIDEO READY")
print("=" * 80)
print("Final video:", final_video)
print("Size:", final_video.stat().st_size, "bytes")
print("Duration seconds:", duration_proc.stdout.strip())

print("\nCell 9 complete.")

RENDERING FINAL HD VIDEO
Command: ffmpeg -y -i /kaggle/working/portrait_video_zipless/outputs/output_raw.mp4 -filter_complex [0:v]scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,gblur=sigma=24,setsar=1[bg];[0:v]scale=1080:1080:force_original_aspect_ratio=decrease,setsar=1[fg];[bg][fg]overlay=(W-w)/2:(H-h)/2,fps=30,format=yuv420p[v] -map [v] -map 0:a? -c:v libx264 -preset medium -crf 18 -c:a aac -b:a 192k -ar 44100 -shortest -movflags +faststart /kaggle/working/portrait_video_zipless/outputs/output.mp4
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --en

In [13]:
# ============================================================
# CELL 10: Preview final video and provide download link
# ============================================================

from pathlib import Path
from IPython.display import Video, FileLink, display

final_video = Path("/kaggle/working/portrait_video_zipless/outputs/output.mp4")
raw_video = Path("/kaggle/working/portrait_video_zipless/outputs/output_raw.mp4")

if final_video.exists() and final_video.stat().st_size > 0:
    print("Final MP4 ready:")
    print(final_video)
    print("Size:", final_video.stat().st_size, "bytes")
    display(Video(str(final_video), embed=True, width=720))
    display(FileLink(str(final_video), result_html_prefix="Download final video: "))
elif raw_video.exists() and raw_video.stat().st_size > 0:
    print("Final HD MP4 missing, but raw SadTalker video exists:")
    print(raw_video)
    display(Video(str(raw_video), embed=True, width=512))
    display(FileLink(str(raw_video), result_html_prefix="Download raw video: "))
else:
    raise FileNotFoundError("No playable MP4 found. Run Cell 8 and Cell 9 first.")

print("\nCell 10 complete.")

Final MP4 ready:
/kaggle/working/portrait_video_zipless/outputs/output.mp4
Size: 403074 bytes


/kaggle/working/portrait_video_zipless/outputs/output.mp4


Cell 10 complete.
